In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
#Install necessary dependencies and libraries
!pip install selenium
import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.options import Options



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 51.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.7/467.7 kB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.7 MB/s eta 0:00:00


In [4]:
!apt-get update
!apt install chromium-chromedriver

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [119 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [2,118 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 2,351 kB in 2s (1,424 kB/s)
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state informati

In [5]:
# Import the required module
from selenium.webdriver.chrome.options import Options

# Create a new Options instance
chrome_options = Options()

# Add the '--headless' argument to run Chrome in headless mode
chrome_options.add_argument('--headless')

# Add any other options you want, for example:
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Create a new instance of the Chrome driver with the options
driver = webdriver.Chrome(options=chrome_options)

In [6]:

from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [7]:
def scrape_sherni():
    faculty_data = []
    base_url = 'https://sherni.inflibnet.ac.in'
    driver.get(base_url)

    # Enter 'Economics' in the search box
    search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//input[@id='title']")))
    search_box.send_keys('Economics')

    # Click the search button
    search_button = driver.find_element(By.XPATH, "//i[@class='fa fa-search']")
    search_button.click()

    # Wait for the results to load
    sleep(5)

    # Set the displayed results to 25
    limits_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//select[@id='limits']")))
    limits_box.send_keys('25')

    # Wait for the results to load
    sleep(10)

    # Loop through the first 3 pages
    page_num = 1
    for _ in range(3):
        # Extract data from the current page
        faculty_list = driver.find_elements(By.XPATH, "//div[contains(@class, 'col-md-9')]//div[3]//div[1]//div[1]//div[1]//div[1]")
        for faculty in faculty_list:
            wait = WebDriverWait(driver, 3)  # wait up to 3 seconds

            # Prioritize collecting Vidwan ID first
            vidwan_id_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[1]/li[1]")))
            vidwan_id = vidwan_id_element.text

            # Collect other information based on the Vidwan ID
            name_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[1]/h4[1]/strong[1]")))
            name = name_element.text
            job_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[2]/span[1]")))
            job = job_element.text
            field_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[3]/li[1]/b[1]")))
            field = field_element.text
            field_specialization_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[4]/li[1]")))
            field_specialization = field_specialization_element.text
            department_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[5]/li[1]")))
            department = department_element.text
            institution_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[6]/li[1]")))
            institution = institution_element.text
            location_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[7]/li[1]")))
            location = location_element.text

            faculty_data.append([vidwan_id, name, job, field, field_specialization, department, institution, location])

        # Check if there is a next page
        next_page_link = driver.find_elements(By.XPATH, "//div[contains(@class, 'filter-results')]//div[1]//ul[1]//li[1]//a[1]")
        if not next_page_link:
            break  # No more pages, exit the loop

        # Click the 'Next' link
        next_page_link[0].click()
        page_num += 1
        sleep(10)  # Add a delay to avoid overwhelming the server

    return faculty_data

In [8]:
# Call the function to scrape data
sherni_data = scrape_sherni()

# Save data to CSV file
columns = ['Vidwan-ID', 'Name', 'Job', 'Field', 'Field Specialization', 'Department', 'Institution', 'Location']
df = pd.DataFrame(sherni_data, columns=columns)
df.to_csv('/content/drive/My Drive/Predoc Application/IEA/sherni_data.csv', index=False)

In [ ]:
def scrape_sherni():
    faculty_data = []
    base_url = 'https://sherni.inflibnet.ac.in'
    driver.get(base_url)

    # Enter 'Economics' in the search box
    search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//input[@id='title']")))
    search_box.send_keys('Economics')

    # Click the search button
    search_button = driver.find_element(By.XPATH, "//i[@class='fa fa-search']")
    search_button.click()

    # Wait for the results to load
    sleep(10)

    # Set the displayed results to 25
    limits_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//select[@id='limits']")))
    limits_box.send_keys('25')

    # Wait for the results to load
    sleep(10)

    # Extract data from the current page
    faculty_list = driver.find_elements(By.XPATH, "//body/div[@id='page']/div[@class='wrapper']/div[@class='content container']/form[@id='myform']/div[@class='row']/div[@class='col-md-9']/div[@class='filter-results']/div/div[1]/div[1]")

    for faculty in faculty_list:
        wait = WebDriverWait(driver, 10)  # wait up to 10 seconds

        # Prioritize collecting Vidwan ID first
        vidwan_id_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[1]/li[1]")))
        vidwan_id = vidwan_id_element.text

        # Collect other information based on the Vidwan ID
        name_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[1]/h4[1]/strong[1]")))
        name = name_element.text
        job_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[2]/span[1]")))
        job = job_element.text
        field_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[3]/li[1]/b[1]")))
        field = field_element.text
        field_specialization_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[4]/li[1]")))
        field_specialization = field_specialization_element.text
        department_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[5]/li[1]")))
        department = department_element.text
        institution_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[6]/li[1]")))
        institution = institution_element.text
        location_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[7]/li[1]")))
        location = location_element.text

        faculty_data.append([vidwan_id, name, job, field, field_specialization, department, institution, location])

    return faculty_data

In [ ]:
def scrape_sherni():
    faculty_data = []
    base_url = 'https://sherni.inflibnet.ac.in'
    driver.get(base_url)

    # Enter 'Economics' in the search box
    search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//input[@id='title']")))
    search_box.send_keys('Economics')

    # Click the search button
    search_button = driver.find_element(By.XPATH, "//i[@class='fa fa-search']")
    search_button.click()

    # Wait for the results to load
    sleep(10)

    # Set the displayed results to 25
    limits_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//select[@id='limits']")))
    limits_box.send_keys('25')

    # Wait for the results to load
    sleep(10)

    # Extract data from the current page
    faculty_container = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//div[@class='filter-results']")))
    faculty_divs = faculty_container.find_elements(By.XPATH, "./div")

    for faculty_div in faculty_divs:
        faculty_info = faculty_div.find_element(By.XPATH, "./div[1]/div[1]")
        wait = WebDriverWait(driver, 10)  # wait up to 10 seconds

        # Prioritize collecting Vidwan ID first
        vidwan_id_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[1]/li[1]")))
        vidwan_id = vidwan_id_element.text

        # Collect other information based on the Vidwan ID
        name_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[1]/h4[1]/strong[1]")))
        name = name_element.text
        job_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[2]/span[1]")))
        job = job_element.text
        field_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[3]/li[1]/b[1]")))
        field = field_element.text
        field_specialization_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[4]/li[1]")))
        field_specialization = field_specialization_element.text
        department_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[5]/li[1]")))
        department = department_element.text
        institution_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[6]/li[1]")))
        institution = institution_element.text
        location_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[7]/li[1]")))
        location = location_element.text

        faculty_data.append([vidwan_id, name, job, field, field_specialization, department, institution, location])

    return faculty_data

In [ ]:
def scrape_sherni():
    faculty_data = []
    base_url = 'https://sherni.inflibnet.ac.in'
    driver.get(base_url)

    # Enter 'Economics' in the search box
    search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//input[@id='title']")))
    search_box.send_keys('Economics')

    # Click the search button
    search_button = driver.find_element(By.XPATH, "//i[@class='fa fa-search']")
    search_button.click()

    # Wait for the results to load
    sleep(5)

    # Set the displayed results to 25
    limits_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//select[@id='limits']")))
    limits_box.send_keys('25')

    # Wait for the results to load
    sleep(10)
    # Extract data from the current page
    faculty_list = driver.find_elements(By.XPATH, "//div[contains(@class, 'col-md-9')]//div[3]//div[1]//div[1]//div[1]//div[1]")
    for faculty in faculty_list:
        wait = WebDriverWait(driver, 3)  # wait up to 3 seconds

        # Prioritize collecting Vidwan ID first
        vidwan_id_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[1]/li[1]")))
        vidwan_id = vidwan_id_element.text

        # Collect other information based on the Vidwan ID
        name_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[1]/h4[1]/strong[1]")))
        name = name_element.text
        job_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[2]/span[1]")))
        job = job_element.text
        field_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[3]/li[1]/b[1]")))
        field = field_element.text
        field_specialization_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[4]/li[1]")))
        field_specialization = field_specialization_element.text
        department_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[5]/li[1]")))
        department = department_element.text
        institution_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[6]/li[1]")))
        institution = institution_element.text
        location_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[7]/li[1]")))
        location = location_element.text

        faculty_data.append([vidwan_id, name, job, field, field_specialization, department, institution, location])

    return faculty_data

In [ ]:
def scrape_sherni():
    faculty_data = []
    base_url = 'https://sherni.inflibnet.ac.in'
    driver.get(base_url)

    # Enter 'Economics' in the search box
    search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//input[@id='title']")))
    search_box.send_keys('Economics')

    # Click the search button
    search_button = driver.find_element(By.XPATH, "//i[@class='fa fa-search']")
    search_button.click()

    # Wait for the results to load
    sleep(10)

    # Set the displayed results to 25
    limits_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//select[@id='limits']")))
    limits_box.send_keys('25')

    # Wait for the results to load
    sleep(10)

    # Loop through the first 3 pages
    page_num = 1
    for _ in range(3):
        # Extract data from the current page
        faculty_list = driver.find_elements(By.XPATH, "//div[contains(@class, 'col-md-9')]//div[3]//div[1]//div[1]//div[1]//div[1]")
        for faculty in faculty_list:
            wait = WebDriverWait(driver, 10)  # wait up to 3 seconds

            # Prioritize collecting Vidwan ID first
            vidwan_id_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[1]/li[1]")))
            vidwan_id = vidwan_id_element.text

            # Collect other information based on the Vidwan ID
            name_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[1]/h4[1]/strong[1]")))
            name = name_element.text
            job_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[2]/span[1]")))
            job = job_element.text
            field_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[3]/li[1]/b[1]")))
            field = field_element.text
            field_specialization_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[4]/li[1]")))
            field_specialization = field_specialization_element.text
            department_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[5]/li[1]")))
            department = department_element.text
            institution_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[6]/li[1]")))
            institution = institution_element.text
            location_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[7]/li[1]")))
            location = location_element.text

            faculty_data.append([vidwan_id, name, job, field, field_specialization, department, institution, location])

        # Check if there is a next page
        next_page_link = driver.find_elements(By.XPATH, "//div[contains(@class, 'filter-results')]//div[1]//ul[1]//li[1]//a[1]")
        if not next_page_link:
            break  # No more pages, exit the loop

        # Click the 'Next' link
        next_page_link[0].click()
        page_num += 1
        sleep(10)  # Add a delay to avoid overwhelming the server

    return faculty_data

In [ ]:
def scrape_sherni():
    faculty_data = []
    base_url = 'https://sherni.inflibnet.ac.in'
    driver.get(base_url)

    # Enter 'Economics' in the search box
    search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//input[@id='title']")))
    search_box.send_keys('Economics')

    # Click the search button
    search_button = driver.find_element(By.XPATH, "//i[@class='fa fa-search']")
    search_button.click()

    # Wait for the results to load
    sleep(5)

    # Increase the displayed results to 100
    limits_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//select[@id='limits']")))
    limits_box.send_keys('100')

    # Wait for the results to load
    sleep(10)

    # Loop through all the pages
    page_num = 1
    while True:
        # Extract data from the current page
        faculty_list = driver.find_elements(By.XPATH, "//div[contains(@class, 'col-md-9')]//div[3]//div[1]//div[1]//div[1]//div[1]")

        for faculty in faculty_list:
            wait = WebDriverWait(driver, 3)  # wait up to 10 seconds
            vidwan_id_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[1]/li[1]")))
            vidwan_id = vidwan_id_element.text

            name_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[1]/h4[1]/strong[1]")))
            name = name_element.text

            job_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[2]/li[2]/span[1]")))
            job = job_element.text

            field_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[3]/li[1]/b[1]")))
            field = field_element.text

            field_specialization_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[4]/li[1]")))
            field_specialization = field_specialization_element.text

            department_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[5]/li[1]")))
            department = department_element.text

            institution_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[6]/li[1]")))
            institution = institution_element.text

            location_element = wait.until(EC.presence_of_element_located((By.XPATH, ".//ul[7]/li[1]")))
            location = location_element.text

            faculty_data.append([vidwan_id, name, job, field, field_specialization, department, institution, location])


            #vidwan_id = faculty.find_element(By.XPATH, ".//ul[1]/li[1]").text
              #name = faculty.find_element(By.XPATH, ".//ul[2]/li[1]/h4[1]/strong[1]").text
              #job = faculty.find_element(By.XPATH, ".//ul[2]/li[2]/span[1]").text
              #field = faculty.find_element(By.XPATH, ".//ul[3]/li[1]/b[1]").text
              #field_specialization = faculty.find_element(By.XPATH, ".//ul[4]/li[1]").text
              #department = faculty.find_element(By.XPATH, ".//ul[5]/li[1]").text
              #institution = faculty.find_element(By.XPATH, ".//ul[6]/li[1]").text
              #location = faculty.find_element(By.XPATH, ".//ul[7]/li[1]").text

              #faculty_data.append([vidwan_id, name, job, field, field_specialization, department, institution, location])

        # Check if there is a next page
        next_page_link = driver.find_elements(By.XPATH, "//div[contains(@class, 'filter-results')]//div[1]//ul[1]//li[1]//a[1]")
        if not next_page_link:
            break  # No more pages, exit the loop

        # Click the 'Next' link
        next_page_link[0].click()
        page_num += 1
        sleep(10)  # Add a delay to avoid overwhelming the server

    return faculty_data

In [ ]:
# Close the webdriver
driver.quit()